# Initialization and data loading
These are the first steps to load libraries and data. As I work in Colab sometimes, there are also some (commented) steps for running the code in Colab. You need to adjust the data directories below to run the notebook. 

In [ ]:
# # Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Load Python libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, RepeatedKFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import VarianceThreshold, SelectFromModel, SequentialFeatureSelector as SFS
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

In [ ]:
# Load data

# Define the data directory path
data_dir = '/home/nikk/SynologyProjects/kaggle_projects/kaggle_titanic/data'
#data_dir = '/content/drive/MyDrive/Titanic'

# Define Random State
random_state = 12

# Load the training dataset
train_path = os.path.join(data_dir, 'train.csv')
train_df = pd.read_csv(train_path)

# Load the test dataset
test_path = os.path.join(data_dir, 'test.csv')
test_df = pd.read_csv(test_path)

# Data profiling and missing values fixing
At first a quick review on the data is executed, to get a look on a sample of rows and to identify missing and invalid values.

In [ ]:
# Quick review of training data
print('\nTraining data review:\n')
print(train_df.head(10))

# Review data types, nulls, min-max values (to help us find invalid values) on training dataset
numeric_columns = train_df.select_dtypes(include=['int64', 'float64']).columns
profiling_ds = pd.DataFrame({
    'data_type': train_df.dtypes,
    'null_pct': train_df.isnull().mean() * 100,
    'min_numeric_value': train_df[numeric_columns].min(),
    'max_numeric_value': train_df[numeric_columns].max()
})
print("\nProfiling analysis on training data:\n")
print(profiling_ds)

# Review data types, nulls, min-max values (to help us find invalid values) on test dataset
numeric_columns = numeric_columns.drop('Survived')
profiling_ds = pd.DataFrame({
    'data_type': test_df.dtypes,
    'null_pct': test_df.isnull().mean() * 100,
    'min_numeric_value': test_df[numeric_columns].min(),
    'max_numeric_value': test_df[numeric_columns].max()
})
print("\nProfiling analysis on testing data:\n")
print(profiling_ds)

We have 4 features that need some type of correction:
1. **Embarked** => As only a few values are missing, we go with MODE.
2. **Cabin** => We will consider null values as an indication that the passenger didn't have a cabin. There are cases that under the same ticket some passengers do have a cabin and some don't (e.g. tickets 2668, PC 17608, 113781). As this happens almost always for 1st class ticket, we will assume that some high class families had their servants with them (but didn't share their cabin). We will also extract the **Deck** of the cabin.
3. **Age** => As 20% of the values are missing, we will try to build a linear model to fill the gaps. For this reason we will create a new variable called **Title**, which will extract out of the Name feature.
4. **Fare** => We consider invalid both null and zero values. We will fix the values by using the average fare on a combination of features that seem logical to have an impact on the ticket value (Pclass, Sex and if the passenger has a cabin or not).

The same ticket can exist in both training and testing datasets (e.g. PC 17599). For this reason, we will build a work on a unified dataset (to secure that any grouping will work correctly). At later steps we will split the dataset into training and testing again.

In [ ]:
# Extract combined dataset
tr = train_df.drop(['Survived'], axis=1)
tr['IsTrain'] = 1
ts = test_df.copy()
ts['IsTrain'] = 0
combined_ds = pd.concat([tr, ts], ignore_index=False)

In [ ]:
# Fix null values for Embarked
combined_ds['Embarked'] = combined_ds['Embarked'].fillna(combined_ds['Embarked'].mode()[0])
# Create flag for passengers with cabin
combined_ds['HasCabin'] = combined_ds['Cabin'].apply(lambda x: 1 if pd.notna(x) else 0)
combined_ds['Deck'] = combined_ds['Cabin'].apply(lambda x: min(t[0] for t in str(x).split()) if pd.notna(x) else "No_Deck")
# Create Title feature out of the Name
combined_ds['Title'] = combined_ds['Name'].apply(lambda x: x.split(',')[1]).str.extract(' ([A-Za-z]+).', expand=False)
combined_ds['Title'] = combined_ds['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Sir', 'Rev', 'Jonkheer', 'Dona', 'the'], 'Entitled')
combined_ds['Title'] = combined_ds['Title'].replace('Mlle', 'Miss')
combined_ds['Title'] = combined_ds['Title'].replace('Ms', 'Miss')
combined_ds['Title'] = combined_ds['Title'].replace('Mme', 'Mrs')
# Fix Fare values
combined_ds['Fare'] = combined_ds['Fare'].apply(lambda x: x if pd.notna(x) and x!=0 else None) # turn 0 to None, so that it will be ignored by Mean
combined_ds['estFare'] = combined_ds.groupby(['Pclass', 'Sex', 'HasCabin'])['Fare'].transform(lambda x: round(x.median(),4))
combined_ds['Fare'] = combined_ds.apply(lambda row: row['estFare'] if pd.isna(row['Fare']) else row['Fare'], axis=1)
# Turn Sex values to numeric
label_sex = {'male':0, 'female':1}
combined_ds['SexFlg'] = combined_ds['Sex'].map(label_sex)

In [ ]:
# Build a simple linear model, using features with defined Age
scaler = StandardScaler()
age_model_cat_features = ['Pclass', 'Sex', 'Embarked', 'Title', 'HasCabin']
age_model_num_features = ['SibSp', 'Parch', 'Fare']
age_model_features = age_model_cat_features + age_model_num_features
age_model_X = combined_ds[combined_ds['Age'].notna()][age_model_features]
age_model_y = combined_ds[combined_ds['Age'].notna()]['Age']
age_model_X = pd.get_dummies(age_model_X, columns=['Pclass', 'Sex', 'Embarked', 'Title'], drop_first=False, dtype=int)
age_model_X[age_model_num_features] = scaler.fit_transform(age_model_X[age_model_num_features])
X_train, X_test, y_train, y_test = train_test_split(age_model_X, age_model_y, test_size=0.3, random_state=random_state)
model = LinearRegression()
model.fit(X_train, y_train)

# Apply the predictions into the combined dataset and fix null Age values
X = pd.get_dummies(combined_ds[age_model_features], columns=['Pclass', 'Sex', 'Embarked', 'Title'], drop_first=False, dtype=int)
X[age_model_num_features] = scaler.fit_transform(X[age_model_num_features])
combined_ds['estAge'] = model.predict(X).round()
combined_ds['Age'] = combined_ds.apply(lambda row: row['estAge'] if pd.isna(row['Age']) else row['Age'], axis=1)

# Data exploration
At this step we go through some data visualizations to get a better insight on the data. We break our features into numerical and categorical and use different visualizations for each case. Some major findings of this task are:
- Small family or company had better chances of survival than traveling alone or in a big group/family.
- A lower fare or class seems to correlate with lower chances of survival.
- Females (especially those in families) had a higher chance to survive.
- Having a cabin was also a good thing!

In [ ]:
# Create exploration dataset
cat_features = ['Pclass', 'Sex', 'Embarked', 'Title', 'HasCabin', 'Deck']
num_features = ['SibSp', 'Parch', 'Age', 'Fare']
features = cat_features + num_features + ['PassengerId']
explore_ds = combined_ds[combined_ds['IsTrain']==1][features]
explore_ds['Survived'] = train_df['Survived']
target_name = 'Survived'
target = train_df[target_name]

In [ ]:
# Visual exploration of numerical features
for i in num_features:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    sns.histplot(explore_ds[i], kde=True, ax=axes[0])
    axes[0].set_title(f"{i} Distribution")
    sns.countplot(data=explore_ds, x=i, hue='Survived', ax=axes[1])
    axes[1].set_title(f"{i} Counts by Survival")
    axes[1].set_xlabel(i)
    axes[1].set_ylabel('Count')
    sns.boxplot(data=explore_ds, x='Survived', y=i, ax=axes[2])
    axes[2].set_title(f"{i} Boxplot by Survival")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis for numerical features
correlation_matrix = explore_ds[num_features].corr()
plt.figure(figsize=(len(num_features) * 2, len(num_features)))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Numerical Features Correlations', y=1.02)
plt.show()

# Visualizing relationships
sns.pairplot(explore_ds[num_features], diag_kind='kde', plot_kws={'alpha':0.7})
plt.suptitle('Numerical Features Scatter Plot Matrix', y=1.02)
plt.show()

In [ ]:
# Create pivot table and barcharts for categorical features
n_features = len(cat_features)
n_rows = (n_features + 2) // 2
n_cols = 4
fig = plt.figure(figsize=(20, 4 * n_rows))

for i, feature in enumerate(cat_features):
    row = i // 2
    col = (i % 2) * 2
    ax_table = plt.subplot2grid((n_rows, n_cols), (row, col))
    ax_bar = plt.subplot2grid((n_rows, n_cols), (row, col + 1))
    pivot = pd.crosstab(explore_ds[feature], target, margins=True)
    ax_table.axis('tight')
    ax_table.axis('off')
    table_data = []
    for idx in pivot.index:
        row_data = [str(idx)] + [str(val) for val in pivot.loc[idx].values]
        table_data.append(row_data)
    
    col_labels = [feature + ' \\ ' + target_name] + [str(col) for col in pivot.columns]
    table = ax_table.table(cellText=table_data,
                           colLabels=col_labels,
                           cellLoc='center',
                           loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 2)
    pivot_no_margin = pd.crosstab(explore_ds[feature], target)
    pivot_no_margin.plot(kind='bar', ax=ax_bar, width=0.8)
    ax_bar.set_xlabel(feature)
    ax_bar.set_ylabel('Count')
    ax_bar.tick_params(axis='x', rotation=45)
    ax_bar.legend(title=target_name)

plt.tight_layout()
plt.show()

Based on the above, we will:
1. turn some of our categorical features to one-hot variables,
2. build a few more features.

In [ ]:
# Convert to one-hot
combined_ds = pd.get_dummies(combined_ds, columns=['Pclass', 'Embarked', 'Title', 'Deck'], drop_first=False, dtype=int)

In [ ]:
# Add more features
combined_ds['TicketMembers'] = combined_ds.groupby(['Ticket'])['PassengerId'].transform(lambda x: x.count())
combined_ds['FamilySize'] = combined_ds['SibSp'] + combined_ds['Parch'] + 1
combined_ds['IsAlone'] = combined_ds.apply(lambda row: 1 if row['TicketMembers']==1 or row['FamilySize']==1 else 0, axis=1)
combined_ds['IsMother'] = combined_ds.apply(lambda row: 1 if ('Mrs.' in row['Name'] and row['Parch']>0) else 0, axis=1)
combined_ds['LogFare'] = combined_ds['Fare'].apply(lambda x: np.log(x))

# Produce new exploration dataset
cat_features = ['IsAlone', 'IsMother']
num_features = ['SibSp', 'Parch', 'Age', 'LogFare', 'TicketMembers', 'FamilySize']
features = cat_features + num_features + ['PassengerId']
explore_ds = combined_ds[combined_ds['IsTrain']==1][features]
explore_ds['Survived'] = train_df['Survived']

# Feature removal
Having completed our dataset, we are going to investigate and drop variables that offer no value to our modelling process. This consists of 3 main actions:
1. Remove constant/quasi constant varriables (variables with marginal variability).
2. Remove duplicate variables (variables with the exact same values). There isn't actually any such variable in our case.
3. Remove numerical features that correlate strongly with other numerical features by using VIF. Only **FamilySize** falls into this category.


In [ ]:
# Remove features with close to 0 variability and create exploration dataset
explore_ds = combined_ds.drop(['PassengerId', 'Name', 'Sex', 'Ticket', 'Fare', 'Cabin', 'estFare', 'estAge'], axis=1)
var = VarianceThreshold(threshold=0.01)
var.fit(explore_ds)
explore_ds = explore_ds[explore_ds['IsTrain']==1]
drop_features = list(explore_ds.columns[~var.get_support()])
drop_features.append('IsTrain')
explore_ds = explore_ds.drop(drop_features, axis=1)
num_features = ['SibSp', 'Parch', 'Age', 'LogFare', 'TicketMembers', 'FamilySize']
cat_features = [col for col in explore_ds.columns if col not in num_features and col not in target_name]

# Find duplicate variables
duplicate_pairs = []
cols = explore_ds.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if explore_ds[cols[i]].equals(explore_ds[cols[j]]):
            duplicate_pairs.append((cols[i], cols[j]))
print("List of duplicate pairs: \n", duplicate_pairs, "\n")

# Check VIF
# vif_test = explore_ds[num_features].copy()
# vif_data = pd.DataFrame()
# vif_data["Feature"] = list(vif_test.columns)
# vif_data["VIF"] = [variance_inflation_factor(vif_test.values, i) for i in range(vif_test.shape[1])]
# print("VIF calculation: \n", vif_data)

# Extract the list of features to work in modelling
#selected_features = [i for i in explore_ds.columns if i not in ['FamilySize']]

selected_features = explore_ds.columns

# Modeling

### With XGBoost

In [ ]:
model_ds = combined_ds[combined_ds['IsTrain']==1][selected_features]
test_ds = combined_ds[combined_ds['IsTrain']==0][selected_features]
X_train, X_val, y_train, y_val = train_test_split(model_ds, target, test_size=0.3, random_state=random_state)

xgb_clf = XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, subsample=0.8, gamma=2, random_state=random_state, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)
xgb_pred = xgb_clf.predict(X_val)

rkf = RepeatedKFold(n_splits=4, n_repeats=30, random_state=random_state)
scores = cross_val_score(xgb_clf, X_val, y_val, cv=rkf, scoring='accuracy')
print(f"Repeated CV Mean accuracy: {scores.mean()}")

test_predictions = xgb_clf.predict(test_ds)
test_xgb = xgb_clf.predict(test_ds)

In [ ]:
sfs = SFS(
    estimator=RandomForestClassifier(n_estimators=100, random_state=random_state),
    n_features_to_select=0.8,
    direction='forward',
    scoring='roc_auc',
    cv=4, n_jobs=12
)
sfs = sfs.fit(X_train, y_train)
selected_features = sfs.get_feature_names_out()

In [ ]:
model_ds = combined_ds[combined_ds['IsTrain']==1][selected_features]
test_ds = combined_ds[combined_ds['IsTrain']==0][selected_features]
X_train, X_val, y_train, y_val = train_test_split(model_ds, target, test_size=0.3, random_state=random_state)

rf_clf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=random_state)
rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_val)

rkf = RepeatedKFold(n_splits=4, n_repeats=30, random_state=random_state)
scores = cross_val_score(rf_clf, X_val, y_val, cv=rkf, scoring='accuracy')
print(f"Repeated CV Mean accuracy: {scores.mean()}")

test_predictions = rf_clf.predict(test_ds)
test_rf = rf_clf.predict(test_ds)

In [ ]:
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': test_predictions
})

submission.to_csv('../submission.csv', index=False)
print("Submission file created successfully.")

In [ ]:
# from sklearn.linear_model import  LogisticRegression

# X_train, X_test, y_train, y_test = train_test_split(
#     explore_ds[selected_features],
#     target,
#     test_size=0.3,
#     random_state=random_state)

# scaler.fit(X_train)
# sel_ = SelectFromModel(LogisticRegression(C=1, penalty='l1', solver='liblinear', random_state=random_state))
# sel_.fit(scaler.transform(X_train), y_train)
# selected_features = X_train.columns[(sel_.get_support())]

# print(len(selected_features))
# selected_features

In [ ]:
# # Train a Random Forest
# from sklearn.inspection import permutation_importance
# X_train, X_test, y_train, y_test = train_test_split(explore_ds[selected_feat], target, test_size=0.3, random_state=random_state)
# rf = XGBClassifier(n_estimators=100, random_state=random_state)
# rf.fit(X_train, y_train)

# # # Standard importance vs Permutation importance
# # std_imp = pd.DataFrame({
# #     'Feature': X_train.columns,
# #     'Importance': rf.feature_importances_
# # }).sort_values('Importance', ascending=False)

# perm_imp = permutation_importance(rf, X_train, y_train, n_repeats=30, random_state=random_state)
# perm_imp_df = pd.DataFrame({
#     'Feature': X_train.columns,
#     'Importance': perm_imp.importances_mean,
#     'Std': perm_imp.importances_std
# }).sort_values('Importance', ascending=False)

# selected_feat = list(perm_imp_df[perm_imp_df['Importance']>0]['Feature'])
# #selected_feat = list(std_imp[std_imp['Importance']>0]['Feature'])
# print(len(selected_feat))
# selected_feat

In [ ]:
# # Train a Random Forest
# from sklearn.inspection import permutation_importance
# X_train, X_test, y_train, y_test = train_test_split(explore_ds[selected_features], target, test_size=0.3, random_state=random_state)
# rf = RandomForestClassifier(n_estimators=100, random_state=random_state)
# rf.fit(X_train, y_train)

# # # Standard importance vs Permutation importance
# # std_imp = pd.DataFrame({
# #     'Feature': X_train.columns,
# #     'Importance': rf.feature_importances_
# # }).sort_values('Importance', ascending=False)

# perm_imp = permutation_importance(rf, X_train, y_train, n_repeats=30, random_state=random_state)
# perm_imp_df = pd.DataFrame({
#     'Feature': X_train.columns,
#     'Importance': perm_imp.importances_mean,
#     'Std': perm_imp.importances_std
# }).sort_values('Importance', ascending=False)

# # print("Standard Importance:\n", std_imp)
# # print("\nPermutation Importance:\n", perm_imp_df)

# selected_features = list(perm_imp_df[perm_imp_df['Importance']>0]['Feature'])
# #selected_feat = list(std_imp[std_imp['Importance']>0]['Feature'])
# print(len(selected_features))
# selected_features

In [ ]:
selected_features = explore_ds.columns
X_train, X_test, y_train, y_test = train_test_split(explore_ds[selected_features], target, test_size=0.3, random_state=random_state)

sfs = SFS(
    #estimator=RandomForestClassifier(n_estimators=100, random_state=random_state),
    estimator=XGBClassifier(n_estimators=100, random_state=random_state),
    n_features_to_select=0.8,
    direction='backward',
    scoring='roc_auc',
    cv=4, n_jobs=12
)
sfs = sfs.fit(X_train, y_train)
#selected_features = sfs.get_feature_names_out()
selected_features = sfs.get_feature_names_out()

print(len(selected_features))
selected_features

In [ ]:
cmb = pd.DataFrame({'PassengerId': test_df['PassengerId'], 'xgb': test_xgb, 'rf': test_rf})
cmb['x']=np.minimum(test_xgb, test_rf)
cmb

In [ ]:
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': cmb['x']
})

submission.to_csv('../submission.csv', index=False)
print("Submission file created successfully.")